# Tarea B — Temperatura, top-p y entropía de la distribución de salida

MMIA 6013 · Tarea de práctica para el solver del Taller 03 v2

Un modelo de lenguaje produce, para el siguiente token, un vector de logits sobre un vocabulario de seis tokens, t0 a t5, con logits z = [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]. Este notebook estudia cómo la temperatura T remodela esa distribución (Parte 1), cómo el muestreo de núcleo (top-p) la recorta (Parte 2), verifica empíricamente el muestreo con 10 000 muestras y la divergencia KL (Parte 3) y discute los límites T → 0 y T → ∞ (Parte 4). Todas las cifras se calculan y muestran en las salidas de las celdas de código.

## Parte 1 — Softmax con temperatura

Se implementa en NumPy la softmax con temperatura, p_i = exp(z_i/T) / Σ_j exp(z_j/T), de forma numéricamente estable: antes de exponenciar se resta el máximo de los logits, de modo que se evalúa exp((z_i − max_j z_j)/T), lo que evita desbordamientos sin alterar el resultado. Se calcula la distribución para T = 0.5, T = 1 y T = 2, la entropía de Shannon en bits de cada una, H = −Σ_i p_i log2(p_i), y se presentan las tres distribuciones y las tres entropías en una única tabla con cuatro decimales, verificando que cada distribución sume 1. Como muestra la salida siguiente, al aumentar T la probabilidad se reparte de forma más uniforme entre los seis tokens y la entropía crece.

In [1]:
import json
import numpy as np

# ----- Datos de la tarea (Parte 1): vocabulario de seis tokens y logits -----
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
z = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)


def softmax_con_temperatura(logits, T):
    """Softmax con temperatura, numéricamente estable.

    p_i = exp(z_i/T) / sum_j exp(z_j/T)
    Se resta max(z) de los logits antes de exponenciar para evitar overflow:
    exp((z_i - m)/T) / sum_j exp((z_j - m)/T), con m = max(z), es idéntica
    matemáticamente pero estable numéricamente.
    """
    logits = np.asarray(logits, dtype=float)
    m = np.max(logits)                       # estabilidad numérica
    exp_z = np.exp((logits - m) / T)
    return exp_z / np.sum(exp_z)


def entropia_bits(p):
    """Entropía de Shannon en bits: H = -sum p_i log2(p_i)."""
    p = np.asarray(p, dtype=float)
    p_pos = p[p > 0]                         # convención 0*log2(0) = 0
    return float(-np.sum(p_pos * np.log2(p_pos)))


temperaturas = [0.5, 1.0, 2.0]

distribuciones = {}
entropias = {}
sumas = {}

for T in temperaturas:
    p = softmax_con_temperatura(z, T)
    distribuciones[T] = p
    entropias[T] = entropia_bits(p)
    sumas[T] = float(np.sum(p))
    # Verificación: cada distribución debe sumar 1
    assert abs(sumas[T] - 1.0) < 1e-12, f"La distribución con T={T} no suma 1"

# ----- Tabla única: distribuciones (una fila por temperatura) + entropías -----
col_w = 9
header = f"{'T':>5} |" + "".join(f"{tok:>{col_w}}" for tok in tokens)
header += f" | {'suma':>{col_w}} {'H(bits)':>{col_w}}"
sep = "-" * len(header)
lines = [header, sep]
for T in temperaturas:
    fila = f"{T:>5.1f} |" + "".join(f"{v:>{col_w}.4f}" for v in distribuciones[T])
    fila += f" | {sumas[T]:>{col_w}.4f} {entropias[T]:>{col_w}.4f}"
    lines.append(fila)
tabla = "\n".join(lines)

print("Parte 1 — Softmax con temperatura y entropía en bits")
print(tabla)
print("\nVerificación: cada distribución suma 1 (tolerancia 1e-12): OK")

# ----- Guardar todos los números calculados en resultados.json -----
resultados = {
    "logits": {tok: float(v) for tok, v in zip(tokens, z)},
    "temperaturas": [float(T) for T in temperaturas],
    "distribuciones": {
        f"T_{T}": {tok: float(p) for tok, p in zip(tokens, distribuciones[T])}
        for T in temperaturas
    },
    "suma_distribuciones": {f"T_{T}": sumas[T] for T in temperaturas},
    "entropia_bits": {f"T_{T}": entropias[T] for T in temperaturas},
}

with open("resultados.json", "w") as f:
    json.dump(resultados, f, indent=2)

print("\nContenido de resultados.json:")
print(json.dumps(resultados, indent=2))


Parte 1 — Softmax con temperatura y entropía en bits
    T |       t0       t1       t2       t3       t4       t5 |      suma   H(bits)
-----------------------------------------------------------------------------------
  0.5 |   0.8231   0.1114   0.0410   0.0225   0.0020   0.0000 |    1.0000    0.9147
  1.0 |   0.5516   0.2029   0.1231   0.0912   0.0275   0.0037 |    1.0000    1.7998
  2.0 |   0.3583   0.2173   0.1693   0.1457   0.0800   0.0294 |    1.0000    2.2888

Verificación: cada distribución suma 1 (tolerancia 1e-12): OK

Contenido de resultados.json:
{
  "logits": {
    "t0": 2.0,
    "t1": 1.0,
    "t2": 0.5,
    "t3": 0.2,
    "t4": -1.0,
    "t5": -3.0
  },
  "temperaturas": [
    0.5,
    1.0,
    2.0
  ],
  "distribuciones": {
    "T_0.5": {
      "t0": 0.8230654368889799,
      "t1": 0.11138979402363647,
      "t2": 0.04097801517761745,
      "t3": 0.022489211553514077,
      "t4": 0.002040175243227406,
      "t5": 3.73671130246886e-05
    },
    "T_1.0": {
      "t0": 

## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución con T = 1 de la Parte 1 se aplica top-p (nucleus sampling) con p = 0.9: se ordenan los tokens de mayor a menor probabilidad, se conserva el conjunto mínimo cuya probabilidad acumulada alcanza 0.9 —incluyendo el token que cruza el umbral, por lo que la masa del núcleo puede superar ligeramente p— y se renormalizan sus probabilidades para que sumen 1. La salida siguiente reporta el orden, las probabilidades acumuladas, qué tokens sobreviven y la distribución renormalizada; los tokens descartados quedan con probabilidad 0 en el vocabulario completo.

In [2]:
import json
import numpy as np


def softmax_con_temperatura(z, T):
    """Softmax con temperatura p_i = exp(z_i/T) / sum_j exp(z_j/T),
    numéricamente estable (se resta el máximo antes de exponenciar)."""
    z = np.asarray(z, dtype=float)
    z_shift = (z - np.max(z)) / float(T)
    e = np.exp(z_shift)
    return e / np.sum(e)


# ---- Datos de la Tarea B (embebidos literalmente) ----
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)

# ---- Parte 1 recomputada: softmax estable con T = 1.0 ----
T = 1.0
probs = softmax_con_temperatura(logits, T)

# ---- Parte 2: muestreo de núcleo (top-p) con p = 0.9 ----
p_umbral = 0.9

# Ordenar los tokens por probabilidad en orden descendente
orden = np.argsort(-probs, kind="stable")
probs_ordenadas = probs[orden]
acumulada = np.cumsum(probs_ordenadas)

# Conjunto mínimo cuya probabilidad acumulada alcanza 0.9,
# incluyendo el token que cruza el umbral
alcanza = acumulada >= p_umbral
if np.any(alcanza):
    n_conservados = int(np.argmax(alcanza)) + 1
else:
    n_conservados = len(tokens)  # respaldo: la suma total es 1.0 >= 0.9

idx_nucleo = orden[:n_conservados]
tokens_sobrevivientes = [tokens[i] for i in idx_nucleo]
probs_nucleo = probs[idx_nucleo]
suma_nucleo = float(np.sum(probs_nucleo))

# Renormalizar las probabilidades del núcleo para que sumen 1
q_nucleo = probs_nucleo / suma_nucleo

# Vista sobre el vocabulario completo (los eliminados quedan con 0)
dist_final_vocabulario = {tok: 0.0 for tok in tokens}
for tok, v in zip(tokens_sobrevivientes, q_nucleo):
    dist_final_vocabulario[tok] = round(float(v), 4)

# ---- Empaquetar resultados (floats de Python) ----
resultados = {
    "logits": {tokens[i]: float(logits[i]) for i in range(len(tokens))},
    "temperatura": float(T),
    "p_top_p": float(p_umbral),
    "softmax_T_1.0": {tokens[i]: float(probs[i]) for i in range(len(tokens))},
    "softmax_T_1.0_4dec": {
        tokens[i]: round(float(probs[i]), 4) for i in range(len(tokens))
    },
    "orden_descendente": [tokens[i] for i in orden],
    "probabilidades_ordenadas_4dec": {
        tokens[int(orden[i])]: round(float(probs_ordenadas[i]), 4)
        for i in range(len(tokens))
    },
    "probabilidad_acumulada_4dec": {
        tokens[int(orden[i])]: round(float(acumulada[i]), 4)
        for i in range(len(tokens))
    },
    "tokens_sobrevivientes": tokens_sobrevivientes,
    "num_tokens_conservados": int(n_conservados),
    "probabilidad_acumulada_nucleo": suma_nucleo,
    "probabilidad_acumulada_nucleo_4dec": round(suma_nucleo, 4),
    "distribucion_renormalizada": {
        tok: round(float(v), 4) for tok, v in zip(tokens_sobrevivientes, q_nucleo)
    },
    "distribucion_renormalizada_precisa": {
        tok: float(v) for tok, v in zip(tokens_sobrevivientes, q_nucleo)
    },
    "suma_distribucion_renormalizada": float(np.sum(q_nucleo)),
    "distribucion_tras_top_p_vocabulario_completo": dist_final_vocabulario,
}

# ---- Escribir resultados.json y mostrar los mismos números en stdout ----
with open("resultados.json", "w") as f:
    json.dump(resultados, f, indent=2)

print(json.dumps(resultados, indent=2))

print("\nResumen:")
print("Softmax T=1.0 (4 dec):",
      {t: round(float(p), 4) for t, p in zip(tokens, probs)})
print("Tokens sobrevivientes (top-p, p=0.9):", ", ".join(tokens_sobrevivientes))
print("Probabilidad acumulada del núcleo:", round(suma_nucleo, 4))
print("Distribución renormalizada (4 dec):",
      {t: round(float(v), 4) for t, v in zip(tokens_sobrevivientes, q_nucleo)})


{
  "logits": {
    "t0": 2.0,
    "t1": 1.0,
    "t2": 0.5,
    "t3": 0.2,
    "t4": -1.0,
    "t5": -3.0
  },
  "temperatura": 1.0,
  "p_top_p": 0.9,
  "softmax_T_1.0": {
    "t0": 0.5516226378718239,
    "t1": 0.2029306277578035,
    "t2": 0.12308364752983937,
    "t3": 0.09118260875807133,
    "t4": 0.02746367398498595,
    "t5": 0.003716804097476065
  },
  "softmax_T_1.0_4dec": {
    "t0": 0.5516,
    "t1": 0.2029,
    "t2": 0.1231,
    "t3": 0.0912,
    "t4": 0.0275,
    "t5": 0.0037
  },
  "orden_descendente": [
    "t0",
    "t1",
    "t2",
    "t3",
    "t4",
    "t5"
  ],
  "probabilidades_ordenadas_4dec": {
    "t0": 0.5516,
    "t1": 0.2029,
    "t2": 0.1231,
    "t3": 0.0912,
    "t4": 0.0275,
    "t5": 0.0037
  },
  "probabilidad_acumulada_4dec": {
    "t0": 0.5516,
    "t1": 0.7546,
    "t2": 0.8776,
    "t3": 0.9688,
    "t4": 0.9963,
    "t5": 1.0
  },
  "tokens_sobrevivientes": [
    "t0",
    "t1",
    "t2",
    "t3"
  ],
  "num_tokens_conservados": 4,
  "probabilida

## Parte 3 — Comprobación empírica

Se toman 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0) y se comparan, token a token, las frecuencias observadas con las probabilidades teóricas en un gráfico de barras. Se calcula además la divergencia KL en bits de la distribución empírica respecto de la teórica, D_KL(q‖p) = Σ_i q_i log2(q_i / p_i), sumando sobre los tokens con q_i > 0. Como muestra la salida, las barras de frecuencias observadas y teóricas son casi indistinguibles y la KL resulta prácticamente nula: el muestreo reproduce la distribución objetivo y solo persiste la pequeña desviación esperable por el tamaño finito de la muestra.

In [3]:
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Subtask s3 (Parte 3) — Comprobación empírica del muestreo top-p
# Recomputa desde cero: softmax estable (T=1), top-p (p=0.9) renormalizado,
# 10,000 muestras con numpy.random.default_rng(0), frecuencias empíricas,
# gráfico de barras comparativo y divergencia KL en bits.
# ----------------------------------------------------------------------

tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
T = 1.0
p_nucleo = 0.9
n_muestras = 10000
semilla = 0

# --- 1) Softmax numéricamente estable a T = 1.0 ------------------------
z = logits / T
z_shift = z - np.max(z)
exp_z = np.exp(z_shift)
probs_softmax = exp_z / np.sum(exp_z)

# --- 2) Top-p: conjunto mínimo que alcanza p=0.9, renormalizado --------
orden = np.argsort(-probs_softmax)            # orden descendente
probs_ordenadas = probs_softmax[orden]
acumulada = np.cumsum(probs_ordenadas)

k = 1
while k < len(acumulada) and acumulada[k - 1] < p_nucleo:
    k += 1
# k = número mínimo de tokens cuya probabilidad acumulada alcanza 0.9

idx_sobrevivientes = np.sort(orden[:k])
probs_sobrevivientes = probs_softmax[idx_sobrevivientes]
probs_renorm = probs_sobrevivientes / np.sum(probs_sobrevivientes)

# Distribución teórica renormalizada sobre el vocabulario completo
teo_full = np.zeros(len(tokens), dtype=float)
teo_full[idx_sobrevivientes] = probs_renorm

# --- 3) Muestreo: 10,000 muestras con default_rng(0) -------------------
rng = np.random.default_rng(semilla)
muestras = rng.choice(idx_sobrevivientes, size=n_muestras, p=probs_renorm)

conteos = np.bincount(muestras, minlength=len(tokens))
emp_full = conteos / float(n_muestras)

# --- 4) KL en bits: D(emp || teo) = sum_i q_i log2(q_i / p_i), q_i > 0 --
mask = emp_full > 0
kl_bits = float(np.sum(emp_full[mask] * np.log2(emp_full[mask] / teo_full[mask])))

# --- 5) Gráfico de barras: observado vs teórico, lado a lado ------------
x = np.arange(len(tokens))
ancho = 0.38
fig, ax = plt.subplots(figsize=(9, 5.5))
b1 = ax.bar(x - ancho / 2, emp_full, ancho,
            label="Frecuencia observada (empírica)", color="#4C72B0")
b2 = ax.bar(x + ancho / 2, teo_full, ancho,
            label="Probabilidad teórica (top-p renormalizada)", color="#DD8452")
for rect, v in zip(b1, emp_full):
    ax.text(rect.get_x() + rect.get_width() / 2, v + 0.006, f"{v:.4f}",
            ha="center", fontsize=8)
for rect, v in zip(b2, teo_full):
    ax.text(rect.get_x() + rect.get_width() / 2, v + 0.006, f"{v:.4f}",
            ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / Frecuencia")
ax.set_title(
    f"Parte 3 — Observado vs teórico (softmax T={T}, top-p p={p_nucleo}, "
    f"{n_muestras} muestras, semilla {semilla})\n"
    f"KL(emp || teo) = {kl_bits:.6f} bits"
)
ax.legend()
ax.set_ylim(0, max(emp_full.max(), teo_full.max()) * 1.18)
fig.tight_layout()
fig.savefig("parte3_frecuencias_vs_teoricas.png", dpi=120)
plt.close(fig)

# --- 6) Resultados a JSON y stdout --------------------------------------
resultados = {
    "logits": {t: float(v) for t, v in zip(tokens, logits)},
    "temperatura": float(T),
    "p_top_p": float(p_nucleo),
    "softmax_T_1.0": {t: float(v) for t, v in zip(tokens, probs_softmax)},
    "orden_descendente": [tokens[i] for i in orden],
    "probabilidad_acumulada_ordenada": {tokens[i]: float(v)
                                        for i, v in zip(orden, acumulada)},
    "tokens_sobrevivientes": [tokens[i] for i in idx_sobrevivientes],
    "num_tokens_conservados": int(k),
    "probabilidad_acumulada_nucleo": float(acumulada[k - 1]),
    "distribucion_renormalizada": {tokens[i]: float(v)
                                   for i, v in zip(idx_sobrevivientes, probs_renorm)},
    "distribucion_teorica_vocabulario_completo": {t: float(v)
                                                  for t, v in zip(tokens, teo_full)},
    "num_muestras": int(n_muestras),
    "semilla": int(semilla),
    "conteos_observados": {t: int(c) for t, c in zip(tokens, conteos)},
    "frecuencias_observadas": {t: float(v) for t, v in zip(tokens, emp_full)},
    "kl_bits_empirica_vs_teorica": kl_bits,
    "figura": "parte3_frecuencias_vs_teoricas.png",
}

with open("resultados.json", "w") as f:
    json.dump(resultados, f, indent=2)

print(json.dumps(resultados, indent=2))


{
  "logits": {
    "t0": 2.0,
    "t1": 1.0,
    "t2": 0.5,
    "t3": 0.2,
    "t4": -1.0,
    "t5": -3.0
  },
  "temperatura": 1.0,
  "p_top_p": 0.9,
  "softmax_T_1.0": {
    "t0": 0.5516226378718239,
    "t1": 0.2029306277578035,
    "t2": 0.12308364752983937,
    "t3": 0.09118260875807133,
    "t4": 0.02746367398498595,
    "t5": 0.003716804097476065
  },
  "orden_descendente": [
    "t0",
    "t1",
    "t2",
    "t3",
    "t4",
    "t5"
  ],
  "probabilidad_acumulada_ordenada": {
    "t0": 0.5516226378718239,
    "t1": 0.7545532656296273,
    "t2": 0.8776369131594667,
    "t3": 0.968819521917538,
    "t4": 0.9962831959025239,
    "t5": 1.0
  },
  "tokens_sobrevivientes": [
    "t0",
    "t1",
    "t2",
    "t3"
  ],
  "num_tokens_conservados": 4,
  "probabilidad_acumulada_nucleo": 0.968819521917538,
  "distribucion_renormalizada": {
    "t0": 0.5693760554907313,
    "t1": 0.2094617451103304,
    "t2": 0.12704497044632815,
    "t3": 0.09411722895261025
  },
  "distribucion_teorica_

## Parte 4 — Pregunta conceptual

**¿Qué pasa cuando T → 0?** En p_i = exp(z_i/T) / Σ_j exp(z_j/T), dividir los logits entre una temperatura cada vez más pequeña amplifica sus diferencias: el término del token con mayor logit (t0) se mantiene en la escala máxima mientras que el de cualquier otro token, exp((z_i − z_max)/T), tiende a 0. Toda la masa de probabilidad se concentra entonces en t0: la distribución tiende a una delta (masa de Dirac) sobre el token de mayor logit y su entropía tiende a 0 bits. La tabla de la Parte 1 muestra esa tendencia: al reducir T de 2 a 1 y a 0.5 la entropía cae de manera monótona y con T = 0.5 ya es menor que 1 bit, muy por debajo del máximo; extrapolando esa tendencia, en el límite T → 0 la entropía llega a 0.

**¿Por qué T → 0 equivale a la decodificación voraz (greedy)?** La decodificación voraz elige en cada paso, de forma determinista, el token de mayor logit, argmax_i z_i (aquí siempre t0). En el límite T → 0 la softmax asigna probabilidad 1 exactamente a ese token y 0 a los demás, de modo que muestrear de esa distribución produce siempre el argmax: muestreo con T → 0 y greedy son el mismo procedimiento. Es precisamente lo que indica la entropía: con H → 0 no queda incertidumbre ni aleatoriedad y el resultado es determinista. Por eso, en la práctica, «temperatura 0» se implementa directamente como argmax/greedy.

**¿Qué pasa cuando T → ∞?** Dividir los logits entre una temperatura cada vez mayor aplasta sus diferencias: z_i/T → 0 para todo i, luego exp(z_i/T) → 1 para todos los tokens y la distribución tiende a la uniforme sobre el vocabulario, p_i → 1/6. La entropía tiende así al máximo posible para seis tokens, log2(6) ≈ 2.585 bits. La tabla de la Parte 1 confirma la dirección del efecto: al aumentar T de 0.5 a 1 y a 2 la entropía crece de forma monótona y con T = 2 ya se acerca al máximo teórico, sin alcanzarlo porque los logits todavía favorecen ligeramente a t0.

**En resumen**, la temperatura interpola entre un régimen determinista (T → 0: delta en el argmax, H → 0, equivalente a la decodificación voraz) y un régimen uniforme y maximalmente aleatorio (T → ∞: distribución uniforme, H → log2 6), con T = 1 como la distribución original del modelo.